## Data Cleaning – Google Glass & Google+

Cleaning the raw YouTube comment CSVs for Google Glass and Google+. Stadia was already handled separately.

In [ ]:
import pandas as pd
import re

glass_old = pd.read_csv('old_google_glass_youtube_comments_raw.csv')
glass_new = pd.read_csv('recent_google_glass_youtube_comments_raw.csv')
plus_old  = pd.read_csv('old_plus_youtube_comments_raw.csv')
plus_new  = pd.read_csv('recent_plus_youtube_comments_raw.csv')

glass_old['product'] = 'google_glass'
glass_new['product'] = 'google_glass'
plus_old['product']  = 'google_plus'
plus_new['product']  = 'google_plus'

glass_old['period'] = 'before'
glass_new['period'] = 'after'
plus_old['period']  = 'before'
plus_new['period']  = 'after'

df = pd.concat([glass_old, glass_new, plus_old, plus_new], ignore_index=True)
print('combined shape:', df.shape)
print(df[['product','period']].value_counts())

### Clean the comment text

In [ ]:
def clean_comment(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

df['clean_comment'] = df['comment'].apply(clean_comment)
print(df['clean_comment'].head(3))

### Remove duplicates and short comments

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=['clean_comment'])
print(f'dropped {before - len(df)} duplicates')

df['word_count'] = df['clean_comment'].apply(lambda x: len(x.split()))
df = df[df['word_count'] >= 4]
print(f'rows after removing short comments: {len(df)}')

### Drop rows where comment is empty or junk

In [ ]:
df = df[df['clean_comment'].str.strip() != '']
df = df[df['clean_comment'] != 'nan']
df.reset_index(drop=True, inplace=True)
print('final shape:', df.shape)
print(df.head())

### Quick summary

In [ ]:
print(df.groupby(['product','period'])['word_count'].agg(['count','mean']).round(2))

### Save cleaned files

In [ ]:
df.to_csv('cleaned_glass_plus_comments.csv', index=False)
print('saved cleaned_glass_plus_comments.csv')

# also save individually
df[df['product']=='google_glass'].to_csv('cleaned_google_glass_comments.csv', index=False)
df[df['product']=='google_plus'].to_csv('cleaned_google_plus_comments.csv', index=False)
print('done')